# 04. Pedigrees, Ploidy, and Sex Chromosomes

This notebook covers three related topics:

1. How pedigree smoothing is represented.
2. How to run haploid, diploid, missing-ploidy, and mixed-ploidy samples.
3. How to think about chrX, chrY, and mitochondrial DNA.


In [1]:
from pathlib import Path
import os, sys, json, math, shutil, time

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
sys.path.insert(0, str(REPO / 'src'))
FIG_DIR = REPO / 'docs' / 'tutorial_deep' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO / 'benchmark_runs' / 'synth_5mb_2k_0p1x'
OUT_DIR = REPO / 'benchmark_runs' / 'tutorial_deep_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print('repo:', REPO)
print('synthetic data exists:', DATA_DIR.exists())


repo: /home/bonnie/Documents/codex/STITCHV2
synthetic data exists: True


## Pedigree Handling

The CLI exposes `--pedigree-strength`, but the relationship matrix itself is currently passed through the Python API:

```python
StitchPipeline(config).prepare_inputs(samples, pedigree=pedigree_matrix, founder_panel=founder_panel)
```

The pedigree matrix is a sparse CSR matrix with shape `(n_samples, n_samples)`. Row `i` contains relative/parent contributors for sample `i`. After HMM dosage inference, smoothing is:

```text
parent_mean = pedigree @ dosage / row_sum(pedigree)
smoothed = (1 - strength) * dosage + strength * parent_mean
```

This is a post-HMM regularizer. It does not replace read likelihoods, and it should be kept weak unless pedigree relationships are trusted.


In [2]:
from scipy import sparse
from stitchv2.pedigree import smooth_dosage_with_pedigree

raw = np.array([
    [0.0, 1.0, 2.0, 1.0],
    [0.0, 0.0, 2.0, 2.0],
    [2.0, 2.0, 0.0, 0.0],
], dtype=np.float32)
rows = [2, 2]
cols = [0, 1]
values = [1.0, 1.0]
pedigree = sparse.csr_matrix((values, (rows, cols)), shape=(3, 3))
smoothed = smooth_dosage_with_pedigree(raw, pedigree, strength=0.25)
print('raw dosage')
print(pd.DataFrame(raw, index=['parent1','parent2','child']))
print('smoothed dosage, strength=0.25')
print(pd.DataFrame(smoothed, index=['parent1','parent2','child']))


raw dosage
           0    1    2    3
parent1  0.0  1.0  2.0  1.0
parent2  0.0  0.0  2.0  2.0
child    2.0  2.0  0.0  0.0
smoothed dosage, strength=0.25
           0      1    2      3
parent1  0.0  0.750  1.5  0.750
parent2  0.0  0.000  1.5  1.500
child    1.5  1.625  0.5  0.375


## Ploidy Rules

- `--ploidy 2`: default diploid run.
- `--ploidy 1`: haploid run.
- `--ploidy 0`: all variants become missing for affected samples without HMM computation.
- `--ploidy-males 1 --ploidy-females 2`: sex-aware mixed ploidy, using `samples.sex`.
- `--ploidy-males 1 --ploidy-females 0`: chrY-like run where females are missing.

For real sex chromosomes, set `--chromosome` to the actual contig name in your BAM and position table, such as `X`, `chrX`, `Y`, `chrY`, `MT`, or `chrM`.


In [3]:
from stitchv2.config import PipelineConfig
from stitchv2.founders import FounderPanel
from stitchv2.pipeline import StitchPipeline

samples = pd.read_parquet(DATA_DIR / 'samples.parquet').head(6).copy()
positions = pd.read_parquet(DATA_DIR / 'positions.parquet').head(30).copy()
samples['sex'] = ['M', 'F', 'M', 'F', 'XY', 'XX']

base = OUT_DIR / '04_ploidy_examples'
if base.exists():
    shutil.rmtree(base)
base.mkdir(parents=True)
positions_path = base / 'positions.parquet'
positions.to_parquet(positions_path, index=False)

def make_founder():
    return FounderPanel(
        chromosome='chrSynthetic',
        positions=positions['POS'].to_numpy(dtype=np.int64),
        ref=positions['REF'].astype(str).to_numpy(),
        alt=positions['ALT'].astype(str).to_numpy(),
        alt_prob=np.full((4, len(positions)), 0.5, dtype=np.float32),
        immutable_mask=np.zeros(4, dtype=bool),
    )

def run_ploidy_case(name, ploidy=2, ploidy_males=None, ploidy_females=None):
    out = base / name
    cfg = PipelineConfig(
        chromosome='chrSynthetic',
        positions_path=positions_path,
        output_dir=out,
        n_founders=4,
        block_size=15,
        em_iterations=1,
        hmm_backend='numpy',
        read_mode='read_stream',
        read_stream_backend='python',
        ploidy=ploidy,
        ploidy_males=ploidy_males,
        ploidy_females=ploidy_females,
        write_genotype_posteriors=True,
        write_genotype_calls=True,
        write_transitions=False,
        profile_memory=False,
    )
    StitchPipeline(cfg).prepare_inputs(samples, founder_panel=make_founder())
    resolved = pd.read_parquet(out / 'samples.parquet')[['sample_id','sex','ploidy']]
    calls = pd.read_parquet(out / 'genotype_calls').head(8)
    return resolved, calls

for case, kwargs in {
    'diploid_default': dict(ploidy=2),
    'chrX_like': dict(ploidy=2, ploidy_males=1, ploidy_females=2),
    'chrY_like': dict(ploidy=2, ploidy_males=1, ploidy_females=0),
}.items():
    resolved, calls = run_ploidy_case(case, **kwargs)
    print('\nCASE:', case)
    display(resolved)
    display(calls)



CASE: diploid_default
    sample_id sex  ploidy
0  sample_000   M       2
1  sample_001   F       2
2  sample_002   M       2
3  sample_003   F       2
4  sample_004  XY       2
5  sample_005  XX       2
    sample_id    chromosome  position  genotype_call  block_id
0  sample_000  chrSynthetic       239              1         0
1  sample_000  chrSynthetic       851              1         0
2  sample_000  chrSynthetic      3036              1         0
3  sample_000  chrSynthetic      3312              1         0
4  sample_000  chrSynthetic      6659              1         0
5  sample_000  chrSynthetic     10619              1         0
6  sample_000  chrSynthetic     11296              1         0
7  sample_000  chrSynthetic     28087              1         0

CASE: chrX_like
    sample_id sex  ploidy
0  sample_000   M       1
1  sample_001   F       2
2  sample_002   M       1
3  sample_003   F       2
4  sample_004  XY       1
5  sample_005  XX       2
    sample_id    chromosome  

## Practical Recipes

chrX, males haploid and females diploid:

```bash
stitchv2 run   --samples samples.parquet   --positions chrX_positions.parquet   --chromosome chrX   --ploidy-males 1   --ploidy-females 2   --output-dir runs/chrX   --n-founders 8
```

chrY, males haploid and females missing:

```bash
stitchv2 run   --samples samples.parquet   --positions chrY_positions.parquet   --chromosome chrY   --ploidy-males 1   --ploidy-females 0   --output-dir runs/chrY   --n-founders 8
```

Mitochondrial DNA is usually treated as haploid for many workflows:

```bash
stitchv2 run   --samples samples.parquet   --positions mt_positions.parquet   --chromosome chrM   --ploidy 1   --output-dir runs/chrM   --n-founders 8
```

If your biological assumptions differ, the important thing is to set ploidy to the number of alternate-allele counts you want in GP: `P + 1` posterior classes.
